# Routed Then Baseline Run + Compare

This notebook runs the agentic routed mode and baseline mode in separate cells,
so interrupting one does not restart the other. Then it compares and plots results.

In [ ]:
import os
import json
import random
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from hpc_llm_setup import (
    config_from_env,
    ensure_eoh_src_on_path,
    start_hpc_bridge,
    stop_hpc_bridge,
    test_bridge,
)

plt.style.use('seaborn-v0_8-whitegrid')


In [ ]:
PROJECT_ROOT, EOH_SRC = ensure_eoh_src_on_path()
COMPARE_ROOT = Path(os.getenv('EOH_COMPARE_OUT', './compare_runs')).resolve()
COMPARE_ROOT.mkdir(parents=True, exist_ok=True)

SEED = int(os.getenv('EOH_SEED', '2024'))
SEED_ROOT = COMPARE_ROOT / f'seed_{SEED}'
SEED_ROOT.mkdir(parents=True, exist_ok=True)

SETTINGS = {
    'problem': os.getenv('EOH_PROBLEM', 'bp_online'),
    'pop_size': int(os.getenv('EOH_POP_SIZE', '8')),
    'generations': int(os.getenv('EOH_N_GENERATIONS', '10')),
    'n_proc': int(os.getenv('EOH_N_PROC', str(min(6, os.cpu_count() or 1)))),
    'eval_parallel_instances': int(os.getenv('EOH_EVAL_PARALLEL_INSTANCES', '1')),
    'eva_timeout': int(os.getenv('EOH_EVA_TIMEOUT', '240')),
    'eval_instances_per_gen': int(os.getenv('EOH_EVAL_INSTANCES_PER_GEN', '256')),
    'holdout_instances': int(os.getenv('EOH_HOLDOUT_INSTANCES', '64')),
    'holdout_eval_interval': int(os.getenv('EOH_HOLDOUT_EVAL_INTERVAL', '1')),
    'route_improvement_epsilon': float(os.getenv('EOH_ROUTE_IMPROVEMENT_EPS', '1e-12')),
    'route_warmup_gens': int(os.getenv('EOH_ROUTE_WARMUP_GENS', '2')),
    'route_e1_cooldown': int(os.getenv('EOH_ROUTE_E1_COOLDOWN', '3')),
    'route_e2_recent_k': int(os.getenv('EOH_ROUTE_E2_RECENT_K', '3')),
    'route_use_probabilistic': os.getenv('EOH_ROUTE_USE_PROBABILISTIC', '1') == '1',
    'route_shuffle_operator_order': os.getenv('EOH_ROUTE_SHUFFLE_OPERATOR_ORDER', '0') == '1',
    'route_controller_enabled': os.getenv('EOH_ROUTE_CONTROLLER_ENABLED', '1') == '1',
    'route_controller_window': int(os.getenv('EOH_ROUTE_CONTROLLER_WINDOW', '5')),
    'route_controller_use_critic': os.getenv('EOH_ROUTE_CONTROLLER_USE_CRITIC', '0') == '1',
    'disable_numba': os.getenv('EOH_DISABLE_NUMBA', '1') == '1',
}

os.environ.setdefault('EOH_LOCAL_LLM_TIMEOUT_S', '600')
os.environ.setdefault('EOH_OFFSPRING_RETRIES', '2')

print('PROJECT_ROOT:', PROJECT_ROOT)
print('COMPARE_ROOT:', COMPARE_ROOT)
print('SEED_ROOT:', SEED_ROOT)
print('SETTINGS:', SETTINGS)


In [ ]:
from eoh import eoh
from eoh.utils.getParas import Paras


def _build_shared_seed_if_needed(seed_root: Path, bridge_url: str, model_id: str, settings: dict):
    shared_seed_path = seed_root / 'shared_initial_population.json'
    if shared_seed_path.exists():
        print(f'[shared-seed] using existing: {shared_seed_path}')
        return shared_seed_path

    seed_build_out = seed_root / '_shared_seed_build'
    if seed_build_out.exists():
        shutil.rmtree(seed_build_out)
    seed_build_out.mkdir(parents=True, exist_ok=True)

    random.seed(SEED)
    np.random.seed(SEED)

    paras = Paras()
    paras.set_paras(
        method='eoh',
        problem=settings['problem'],
        llm_use_local=True,
        llm_local_url=bridge_url,
        llm_model=model_id,
        ec_pop_size=settings['pop_size'],
        ec_n_pop=0,
        exp_n_proc=settings['n_proc'],
        exp_output_path=str(seed_build_out),
        exp_debug_mode=False,
        eva_timeout=settings['eva_timeout'],
        eval_parallel_instances=settings['eval_parallel_instances'],
        eval_instances_per_gen=settings['eval_instances_per_gen'],
        holdout_instances=settings['holdout_instances'],
        holdout_eval_interval=settings['holdout_eval_interval'],
        route_improvement_epsilon=settings['route_improvement_epsilon'],
        route_warmup_gens=settings['route_warmup_gens'],
        route_e1_cooldown=settings['route_e1_cooldown'],
        route_e2_recent_k=settings['route_e2_recent_k'],
        route_use_probabilistic=settings['route_use_probabilistic'],
        route_controller_enabled=settings['route_controller_enabled'],
        route_controller_window=settings['route_controller_window'],
        eoh_mode='baseline',
        log_full_population=True,
    )
    if settings.get('disable_numba', False):
        paras.eva_numba_decorator = False

    runner = eoh.EVOL(paras)
    runner.run()

    pop0_path = seed_build_out / 'results' / 'pops' / 'population_generation_0.json'
    if not pop0_path.exists():
        raise RuntimeError(f'missing {pop0_path}')

    with pop0_path.open('r', encoding='utf-8') as f:
        pop0 = json.load(f)

    seeds = []
    for ind in pop0:
        if not isinstance(ind, dict):
            continue
        code = ind.get('code')
        algorithm = ind.get('algorithm')
        if isinstance(code, str) and isinstance(algorithm, str):
            seeds.append({'algorithm': algorithm, 'code': code})

    if len(seeds) == 0:
        raise RuntimeError('no valid seed algorithms built')

    with shared_seed_path.open('w', encoding='utf-8') as f:
        json.dump(seeds, f, indent=2)

    print(f'[shared-seed] created: {shared_seed_path} ({len(seeds)} seeds)')
    return shared_seed_path


def _run_single_mode(mode: str, mode_out: Path, bridge_url: str, model_id: str, settings: dict, shared_seed_path: Path, use_critic: bool | None = None, shuffle_operator_order: bool | None = None):
    if mode_out.exists():
        shutil.rmtree(mode_out)
    mode_out.mkdir(parents=True, exist_ok=True)

    random.seed(SEED)
    np.random.seed(SEED)

    paras = Paras()
    if use_critic is None:
        use_critic = bool(settings.get('route_controller_use_critic', False))
    if shuffle_operator_order is None:
        shuffle_operator_order = bool(settings.get('route_shuffle_operator_order', False))

    paras.set_paras(
        method='eoh',
        problem=settings['problem'],
        llm_use_local=True,
        llm_local_url=bridge_url,
        llm_model=model_id,
        ec_pop_size=settings['pop_size'],
        ec_n_pop=settings['generations'],
        exp_n_proc=settings['n_proc'],
        exp_output_path=str(mode_out),
        exp_debug_mode=False,
        eva_timeout=settings['eva_timeout'],
        eval_parallel_instances=settings['eval_parallel_instances'],
        eval_instances_per_gen=settings['eval_instances_per_gen'],
        holdout_instances=settings['holdout_instances'],
        holdout_eval_interval=settings['holdout_eval_interval'],
        route_improvement_epsilon=settings['route_improvement_epsilon'],
        route_warmup_gens=settings['route_warmup_gens'],
        route_e1_cooldown=settings['route_e1_cooldown'],
        route_e2_recent_k=settings['route_e2_recent_k'],
        route_use_probabilistic=settings['route_use_probabilistic'],
        route_shuffle_operator_order=bool(shuffle_operator_order),
        route_controller_enabled=settings['route_controller_enabled'],
        route_controller_window=settings['route_controller_window'],
        route_controller_use_critic=bool(use_critic),
        exp_use_seed=True,
        exp_seed_path=str(shared_seed_path),
        eoh_mode=mode,
        log_full_population=False,
    )
    if settings.get('disable_numba', False):
        paras.eva_numba_decorator = False

    print(f'[run] mode={mode} use_critic={bool(use_critic)} shuffle_operator_order={bool(shuffle_operator_order)} out={mode_out}')
    runner = eoh.EVOL(paras)
    runner.run()
    print(f'[run] done mode={mode}')


def run_mode_with_bridge(mode: str, use_critic: bool | None = None, shuffle_operator_order: bool | None = None, output_suffix: str | None = None):
    cfg = config_from_env()
    server = None
    try:
        server, _thread, bridge_url, model_id = start_hpc_bridge(cfg)
        status, payload = test_bridge(bridge_url)
        print('[bridge] status:', status, '| payload:', payload)

        shared_seed_path = _build_shared_seed_if_needed(SEED_ROOT, bridge_url, model_id, SETTINGS)
        mode_name = mode if output_suffix is None else f"{mode}_{output_suffix}"
        mode_out = SEED_ROOT / mode_name
        _run_single_mode(mode, mode_out, bridge_url, model_id, SETTINGS, shared_seed_path, use_critic=use_critic, shuffle_operator_order=shuffle_operator_order)
    finally:
        stop_hpc_bridge(server)
        print('[bridge] stopped')


## Run 1A: Routed Ordered (No Shuffle)

Run this cell first. If interrupted, rerun this cell only.

In [ ]:
run_mode_with_bridge('routed', use_critic=False, shuffle_operator_order=False, output_suffix='ordered')


## Run 1B: Routed Shuffled (Ablation)

Run this cell after ordered routed. This tests shuffled operator execution order.

In [ ]:
run_mode_with_bridge('routed', use_critic=False, shuffle_operator_order=True, output_suffix='shuffled')


## Run 2: Baseline

Run this cell after routed. If interrupted, rerun this cell only.

In [ ]:
run_mode_with_bridge('baseline', use_critic=False, shuffle_operator_order=False)


In [ ]:
def read_jsonl(path: Path):
    if not path.exists():
        return []
    rows = []
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except Exception:
                pass
    return rows


def load_mode_df(seed_root: Path, mode: str):
    p = seed_root / mode / 'results' / 'run_log.jsonl'
    return pd.DataFrame(read_jsonl(p))


base_df = load_mode_df(SEED_ROOT, 'baseline')
routed_ordered_df = load_mode_df(SEED_ROOT, 'routed_ordered')
routed_shuffled_df = load_mode_df(SEED_ROOT, 'routed_shuffled')
routed_df = routed_ordered_df if len(routed_ordered_df) > 0 else load_mode_df(SEED_ROOT, 'routed')

print('baseline rows:', len(base_df))
print('routed_ordered rows:', len(routed_ordered_df))
print('routed_shuffled rows:', len(routed_shuffled_df))
print('routed (active for legacy plots) rows:', len(routed_df))

def _last_best(df):
    if len(df) == 0 or 'best_fitness' not in df.columns:
        return None
    return float(df['best_fitness'].iloc[-1])

print('final best baseline:', _last_best(base_df))
print('final best routed_ordered:', _last_best(routed_ordered_df))
print('final best routed_shuffled:', _last_best(routed_shuffled_df))
display(base_df.tail(5))
display(routed_df.tail(5))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

if not base_df.empty:
    y = base_df['train_fitness'] if 'train_fitness' in base_df.columns else base_df['best_fitness']
    axes[0].plot(base_df['gen'], y, label='baseline')
if not routed_df.empty:
    y = routed_df['train_fitness'] if 'train_fitness' in routed_df.columns else routed_df['best_fitness']
    axes[0].plot(routed_df['gen'], y, label='routed')
axes[0].set_title('Train Fitness vs Generation')
axes[0].set_xlabel('gen')
axes[0].set_ylabel('fitness (lower better)')
axes[0].legend()

if not base_df.empty:
    axes[1].plot(base_df['gen'], base_df['invalid_rate'], label='baseline')
if not routed_df.empty:
    axes[1].plot(routed_df['gen'], routed_df['invalid_rate'], label='routed')
axes[1].set_title('Invalid Rate vs Generation')
axes[1].set_xlabel('gen')
axes[1].set_ylabel('invalid_rate')
axes[1].legend()

if not routed_df.empty and 'chosen_operator' in routed_df.columns:
    op_counts = routed_df['chosen_operator'].value_counts()
    axes[2].bar(op_counts.index.astype(str), op_counts.values)
axes[2].set_title('Routed Operator Counts')
axes[2].set_xlabel('operator')
axes[2].set_ylabel('count')

plt.tight_layout()
plt.show()


In [ ]:
agent_files = [
    'agent_observation.jsonl',
    'agent_diagnosis.jsonl',
    'agent_plan.jsonl',
    'agent_critic.jsonl',
]

for fname in agent_files:
    p = SEED_ROOT / 'routed' / 'results' / fname
    rows = read_jsonl(p)
    print(fname, 'rows=', len(rows), 'path=', p)
    if rows:
        display(pd.DataFrame(rows).tail(3))
